In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import textwrap
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import matplotlib.pyplot as plt

from tackai.models_comparison import (
    calc_regression_metrics,
    make_scatterplot,
)

In [ ]:
# Make plot directory if it doesn't exist
plot_dir = Path("../plots")
plot_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {plot_dir.resolve()}")

proj_dir = Path("/cephyr/users/ribes/Alvis/mimer/stefano/TACK")
predictions_dir = proj_dir / "predictions"
protac_stan_preds_dir = Path("/mimer/NOBACKUP/groups/naiss2023-6-290/nils/PROTAC-STAN/results_multitask_20260119_0248/predictions/")

In [ ]:
def clean_method_name(method: str, data: str = None) -> str:
    """Cleans method names for better visualization."""
    final_method = ''
    if 'protac-stan' in method.lower():
        return 'PROTAC-STAN'
    if 'xgb' in method.lower():
        final_method = 'XGB'
    if 'mlp' in method.lower():
        final_method = 'MLP'

    final_method += '-QR' if '_qr' in method.lower() else ''
    final_method += '-MVE' if '_mve' in method.lower() else ''
    final_method += '-BIN' if '_bin' in method.lower() else ''
    final_method += '-DMAX' if '_dmax' in method.lower() else ''
    final_method += '-DC50' if '_dc50' in method.lower() else ''
    
    data_features = []
    
    data_info = method if data is None else data
    
    if 'fp512r16' in data_info.lower():
        data_features.append('FP')
    if '_desc' in data_info.lower():
        data_features.append('Mol-Desc')
    if '_poi_ord' in data_info.lower():
        data_features.append('POI-Ord')
    if '_lig_ord' in data_info.lower():
        data_features.append('E3-Ord')
    if '_assay_time' in data_info.lower():
        data_features.append('Time')
    if '_cell_ord' in data_info.lower():
        data_features.append('Cell-Ord')
    if '_cell_text' in data_info.lower():
        data_features.append('Cell-Text')
    if '_cell_pt' in data_info.lower():
        data_features.append('Cell-Emb')
    if '_cell_onehot' in data_info.lower():
        data_features.append('Cell-OneHot')
    if '_poi_pt' in data_info.lower():
        data_features.append('POI-Emb')
    if '_poi_onehot' in data_info.lower():
        data_features.append('POI-OneHot')
    if '_poi_vec' in data_info.lower():
        data_features.append('POI-Vec')
    if '_lig_pt' in data_info.lower():
        data_features.append('E3-Emb')
    if '_lig_onehot' in data_info.lower():
        data_features.append('E3-OneHot')
    if '_lig_vec' in data_info.lower():
        data_features.append('E3-Vec')
    if '_poi_emb' in data_info.lower():
        data_features.append('POI-ESM-S')
    if '_lig_emb' in data_info.lower():
        data_features.append('E3-ESM-S')
    if '_poi_pca44_lig_pca7' in data_info.lower():
        data_features.append('POI/E3-ESM-S-PCA')

    # Remove extra spaces
    final_method += ' ' + ' '.join(sorted(data_features))
    return final_method

# Get all files in the predictions directory
prediction_files = list(predictions_dir.glob("*.csv"))
protac_stan_files = list(protac_stan_preds_dir.glob("*.csv"))

# Load all CSV files in the predictions directory via a for loop
method2file = defaultdict(list)
clean_method2file = defaultdict(set)
raw_methods = set()
results = []
for file in sorted(prediction_files):
    # Extract model name from filename
    model_name = file.stem.split("=")[1].split("-")[0]
    data_name = file.stem.split("=")[2].split("-")[0]
    raw_methods.add((model_name + ' ' + data_name, clean_method_name(model_name, data_name)))
    method2file[model_name + ' ' + data_name].append(file.stem)
    clean_method2file[clean_method_name(model_name, data_name)].add(model_name + '-data=' + data_name)

for file in sorted(protac_stan_files):
    model_name = 'PROTAC-STAN'
    raw_methods.add((model_name, clean_method_name(model_name)))
    method2file[model_name].append(file.stem)
    clean_method2file[clean_method_name(model_name, data_name)].add(model_name + '-data=' + data_name)

for raw_method, method in sorted(raw_methods, key=lambda x: x[1]):
    print(f"{method:20} -> {raw_method}")
    
# Check that all the raw_method are unique
raw_method_names = [rm for rm, m in raw_methods]
assert len(raw_method_names) == len(set(raw_method_names)), "Raw method names are not unique!"

# Select the best methods to compare
methods = [
    # 'XGB Cell-Text E3-Ord FP Mol-Desc POI-Vec Time',
    # 'XGB Cell-Ord E3-Ord Mol-Desc POI-Vec Time',
    # 'XGB Cell-Text E3-Ord Mol-Desc POI-Vec Time',
    # 'MLP Cell-Text E3-ESM-S FP Mol-Desc POI-ESM-S POI/E3-ESM-S-PCA Time',
    # 'MLP Cell-Text E3-Ord Mol-Desc POI-Ord Time',
    # 'MLP Cell-Ord E3-Ord Mol-Desc POI-Vec Time',

    # 'XGB-DC50 Cell-Text E3-Ord FP Mol-Desc POI-Vec Time',
    'XGB-DC50 Cell-Text E3-OneHot FP Mol-Desc POI-Vec Time',
    'MLP-DC50 Cell-OneHot E3-OneHot Mol-Desc POI-OneHot Time',
    # 'XGB-DMAX Cell-Text E3-Ord FP Mol-Desc POI-Vec Time',
    'XGB-DMAX Cell-Text E3-OneHot Mol-Desc POI-Vec Time',
    # 'XGB-DMAX Cell-Text E3-Ord Mol-Desc POI-Vec Time',
    'MLP-DMAX Cell-OneHot E3-OneHot Mol-Desc POI-OneHot Time',
    'XGB-BIN Cell-Text E3-OneHot FP Mol-Desc POI-Vec Time',
    'XGB-BIN Cell-OneHot E3-OneHot Mol-Desc POI-Vec Time',
    # 'XGB-BIN Cell-Ord E3-Ord Mol-Desc POI-Vec Time',
    # 'XGB-BIN Cell-Text E3-Ord FP Mol-Desc POI-Vec Time',
    # 'XGB-BIN Cell-Text E3-Ord Mol-Desc POI-Vec Time',
    'MLP-BIN Cell-Emb E3-Emb Mol-Desc POI-Emb Time',
    'MLP-BIN Cell-Emb E3-Emb FP POI-Emb Time',
    'MLP-BIN Cell-OneHot E3-OneHot Mol-Desc POI-OneHot Time',
]

for method in methods:
    if method not in [m for rm, m in raw_methods]:
        raise ValueError(f"Method {method} not found in raw methods!")

# Check that all selected methods have the same number of files
num_files = [len(method2file[rm]) for rm, m in raw_methods if m in methods]
if len(set(num_files)) != 1:
    raise ValueError("Not all selected methods have the same number of files!")

print()
for m in methods:
    files = clean_method2file[m]
    for f in files:
        print(f"{m:40} -> {f}")

In [ ]:
# Load all CSV files in the predictions directory via a for loop
results = []
for file in tqdm(prediction_files, desc="Loading prediction files"):
    # Extract model name from filename
    model_name = file.stem.split("=")[1].split("-")[0]
    data_name = file.stem.split("=")[2].split("-")[0]
    
    clean_method = clean_method_name(model_name, data_name)
    if clean_method not in methods:
        continue
    
    df = pd.read_csv(file)
    df['method'] = clean_method

    # Rename columns for consistency
    df = df.rename(columns={'group': 'split', 'value_type': 'task', 'confidence': 'prob'})
    # Rename task column values, from 'dmax' to 'Dmax' and 'dc50' to 'DC50'
    df['task'] = df['task'].str.replace('dmax', 'Dmax')
    df['task'] = df['task'].str.replace('dc50', 'DC50')
    df['task'] = df['task'].str.replace('binary_class', 'bin')
    df['task'] = df['task'].str.replace('heldout', 'bin')

    # For XGBoost, the 'pred' column refers to probabilities, so we need to
    # rename it and then threshold it at 0.5 to get binary predictions
    if 'bin' in df['task'].unique()[0]:
        if 'prob' not in df.columns:
            df['prob'] = df['pred'].copy()
            df['pred'] = (df['prob'] >= 0.5).astype(int)

    if 'PROTAC-STAN' in file.stem:
        df['set'] = 'test' if 'heldout' in file.stem else 'val'
    else:
        df['set'] = 'test' if 'test' in file.stem else 'val'
    results.append(df)

results_df = pd.concat(results, ignore_index=True)

# For get the maximum number of folds for any method
max_folds = results_df.groupby('method')['fold'].nunique().max()

# Build a mask to keep only (method, task) pairs with the required number of folds
mask = []
for (method, task), group in results_df.groupby(['method', 'task']):
    num_folds = group['fold'].nunique()
    required_folds = max_folds
    if num_folds < max_folds:
        print(f"WARNING: Method '{method}' for task {task} has only {num_folds} folds (expected {required_folds})")
        mask.extend(group.index.tolist())

# Remove only the problematic (method, task) pairs
results_df = results_df.drop(mask).reset_index(drop=True)

# Print all methods
print("\nMethods found in results:")
for method in sorted(results_df['method'].unique()):
    print(f"- {method}")

def dc50_to_pdc50(x):
    """Convert DC50 in nM to pDC50."""
    return -np.log10(x * 1e-9 + 1e-12)

# Convert all task 'DC50' to pDC50 values by taking -log10, they are in nM
for task_group, group in results_df.groupby('task'):
    if task_group == 'DC50':
        results_df.loc[group.index, 'target'] = group['target'].apply(dc50_to_pdc50)
        results_df.loc[group.index, 'pred'] = group['pred'].apply(dc50_to_pdc50)
        
        if 'pred_lower' in results_df:
            results_df.loc[group.index, 'pred_lower'] = group['pred_lower'].apply(dc50_to_pdc50)
            results_df.loc[group.index, 'pred_upper'] = group['pred_upper'].apply(dc50_to_pdc50)

In [ ]:
print(results_df['set'].unique())
print(results_df['task'].unique())
print(results_df['pred'].unique())
if 'prob' in results_df.columns:
    print(results_df['prob'].unique())

In [ ]:
# NOTE: The DeepPROTACs reports cutoffs of 80% for Dmax and 100nM for DC50
classification_cutoffs = {
    'DC50': dc50_to_pdc50(100),  # 100 nM -> ~7 in pDC50
    'Dmax': 80.0, # 80%
}

In [ ]:
# def wrap_method_names(method_name, width=10):
#     """ Wrap method names to a given width for better visualization."""
#     return textwrap.fill(method_name, width)

# dmax_methods = {
#     # # 'XGB Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'XGB Cell-Ord E3-Ord POI-Vec Mol-Desc',
#     # 'XGB Cell-Desc E3-Ord Mol-Desc POI-Vec Time': 'XGB Cell-Desc E3-Ord POI-Vec Mol-Desc',
#     # 'MLP Cell-Ord E3-Ord Mol-Desc POI-Vec Time': 'MLP Cell-Ord E3-Ord POI-Vec Mol-Desc',
#     'XGB-DMAX Cell-Text E3-Ord Mol-Desc POI-Vec Time': 'XGB Cell-Text E3-Ord Mol-Desc POI-Vec',
#     'MLP-DMAX Cell-OneHot E3-OneHot Mol-Desc POI-OneHot Time': 'MLP Cell-OneHot E3-OneHot Mol-Desc POI-OneHot',
# }
# dc50_methods = {
#     # 'XGB Cell-Desc E3-Ord FP Mol-Desc POI-Vec Time': 'XGB Cell-Desc E3-Ord POI-Vec Mol-Desc FP',
#     # 'MLP Cell-Desc E3-ESM-S FP Mol-Desc POI-ESM-S POI/E3-ESM-S-PCA Time': 'MLP Cell-Desc E3/POI-ESM-S (PCA) Mol-Desc FP',
#     # # 'MLP Cell-Desc E3-Ord Mol-Desc POI-Ord Time': 'MLP Cell-Desc E3-Ord Mol-Desc POI-Ord',
#     'XGB-DC50 Cell-Text E3-OneHot FP Mol-Desc POI-Vec Time': 'XGB Cell-Text E3-OneHot FP Mol-Desc POI-Vec',
#     'MLP-DC50 Cell-OneHot E3-OneHot Mol-Desc POI-OneHot Time': 'MLP Cell-OneHot E3-OneHot Mol-Desc POI-OneHot',
# }

# for task in ['Dmax', 'DC50']:
#     for dset in ['test']:
#         subset = results_df[results_df['set'] == dset].copy()
#         subset = subset[subset['task'] == task]
#         if subset.empty:
#             print(f"WARNING: No data for task {task} on set {dset}, skipping...")
#             continue
        
#         # Filter only the relevant methods for the current task
#         # Also replace method names according to the mapping
#         if task == 'Dmax':
#             subset = subset[subset['method'].isin(dmax_methods.keys())]
#             subset['method'] = subset['method'].map(dmax_methods)
#         elif task == 'DC50':
#             subset = subset[subset['method'].isin(dc50_methods.keys())]
#             subset['method'] = subset['method'].map(dc50_methods)

#         # # Wrap method names for better visualization
#         # subset['method'] = subset['method'].apply(lambda x: wrap_method_names(x, width=20))

#         print("=" * 80)
#         print(f"Analysis of {task} models on {dset} set:")
#         print("=" * 80)
#         make_scatterplot(subset, "target", "pred", thresh=classification_cutoffs[task], cycle_col="fold", group_col="method")

#         # plt.suptitle(f"{task} models on {dset} set", fontsize=16, y=1.02)
#         plt.show()

In [ ]:
colors = {
    'blue': '#4B9ECE',
    'orange': '#FFAA6E',
    'light_blue': '#50B1D8',
    'dark_orange': '#FF8428',
    'green': '#9DCE9C',
    'purple': '#C8ABDA',
}

In [ ]:
def set_size(width_pt, fraction=1, subplots=(1, 1)):
    """
    Set figure dimensions to avoid scaling in LaTeX.
    
    Parameters
    ----------
    width_pt: float
            Document width in points (result of \the\columnwidth)
    fraction: float, optional
            Fraction of the width which you wish the figure to occupy
    subplots: array-like, optional
            The number of rows and columns of subplots.
    
    Returns
    -------
    fig_dim: tuple
            Dimensions of figure in inches
    """
    # Width of figure (in pts)
    fig_width_pt = width_pt * fraction

    # Convert from pt to inches
    inches_per_pt = 1 / 72.27

    # Golden ratio to set aesthetic height
    golden_ratio = (5**.5 - 1) / 2

    # Figure width in inches
    fig_width_in = fig_width_pt * inches_per_pt
    
    # Figure height in inches
    fig_height_in = fig_width_in * golden_ratio * (subplots[0] / subplots[1])

    return (fig_width_in, fig_height_in)


def make_scatterplot_pair(df_left, df_right, val_col, pred_col, thresh_left, thresh_right, 
                          cycle_col="cv_cycle", group_col="method", axis_left="Dmax", axis_right="pDC50",
                          # LaTeX sizing parameters
                          tex_width=506.295, fraction=1.0, use_constrained_layout=True,
                          # Font parameters
                          font_family='sans-serif', font_size=9, 
                          xlabel_fontweight='bold', tick_font_size=None,
                          # Scatter plot parameters
                          scatter_color='C0', scatter_alpha=0.3, scatter_size=5,
                          # Line parameters
                          diagonal_color='black', diagonal_style='--', diagonal_width=1,
                          threshold_color='red', threshold_style='--', threshold_width=1, threshold_alpha=0.7,
                          # Other parameters
                          grid_alpha=0.3, metrics_box_alpha=0.8, subplot_label='a)'):
    """
    Create side-by-side scatter plots (parity plots) for two tasks without marginal distributions.
    Optimized for LaTeX/ACM publication format.

    Parameters:
    df_left (pd.DataFrame): Dataframe for the first task (left subplot).
    df_right (pd.DataFrame): Dataframe for the second task (right subplot).
    val_col (str): The column name for the ground truth values.
    pred_col (str): The column name for the predicted values.
    thresh_left (float): Threshold for binary classification (first task).
    thresh_right (float): Threshold for binary classification (second task).
    cycle_col (str): The column name indicating the cross-validation fold.
    group_col (str): The column name indicating the groups/methods.
    axis_left (str): Label for the first task axis.
    axis_right (str): Label for the second task axis.
    
    LaTeX sizing parameters:
    tex_width (float): LaTeX text/column width in points. Default 506.295pt (ACM text width).
    fraction (float): Fraction of tex_width to use for figure. Default 1.0.
    use_constrained_layout (bool): Use constrained layout (recommended for LaTeX). Default True.
    
    Font parameters:
    font_family (str): Font family for all text elements.
    font_size (int): Base font size (should match LaTeX body font, typically 9 for ACM).
    xlabel_fontweight (str): Font weight for x/y-axis labels.
    tick_font_size (int): Font size for tick labels. If None, uses font_size.
    
    Scatter plot parameters:
    scatter_color (str): Color for scatter points.
    scatter_alpha (float): Transparency for scatter points.
    scatter_size (int): Size of scatter points.
    
    Line parameters:
    diagonal_color (str): Color for diagonal reference line.
    diagonal_style (str): Line style for diagonal line.
    diagonal_width (float): Line width for diagonal line.
    threshold_color (str): Color for threshold lines.
    threshold_style (str): Line style for threshold lines.
    threshold_width (float): Line width for threshold lines.
    threshold_alpha (float): Transparency for threshold lines.
    
    Other parameters:
    grid_alpha (float): Transparency for grid lines.
    metrics_box_alpha (float): Transparency for metrics text box.
    subplot_label (str): Label for the first subplot (e.g., 'a)', 'b)').

    Returns:
    fig, axes
    """
    # Set default tick font size if not specified
    if tick_font_size is None:
        tick_font_size = font_size - 2
    xylabel_font_size = font_size
    
    # Calculate metrics for both tasks
    df_split_metrics = [
        calc_regression_metrics(df_left, cycle_col=cycle_col, val_col=val_col, pred_col=pred_col, thresh=thresh_left),
        calc_regression_metrics(df_right, cycle_col=cycle_col, val_col=val_col, pred_col=pred_col, thresh=thresh_right),
    ]

    # Set font globally
    plt.rcParams['font.family'] = font_family
    
    # Create figure with LaTeX-compatible sizing
    figsize = set_size(tex_width, fraction=fraction, subplots=(1, 2))
    w, h = figsize
    figsize = (w, h * 1.5)  # Increase height slightly to accommodate metrics box
    # figsize = (8, 4)
    
    if use_constrained_layout:
        fig, axes = plt.subplots(1, 2, figsize=figsize, layout='constrained')
    else:
        fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    for i, (ax, thresh, df_method, df_metrics, ax_name) in enumerate(
        zip(axes, [thresh_left, thresh_right], [df_left, df_right], df_split_metrics, [axis_left, axis_right])
    ):
        # Calculate classification metrics
        y_true = df_method[val_col] > thresh
        y_pred = df_method[pred_col] > thresh
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        roc_auc = roc_auc_score(y_true, y_pred)

        # ===== Main scatter plot =====
        scatter_color = colors['blue'] if i == 0 else colors['orange']
        
        ax.scatter(df_method[pred_col], df_method[val_col], 
                  alpha=scatter_alpha, s=scatter_size, color=scatter_color,
                #   edgecolor='black',
                  rasterized=True,
                  linewidths=0.1)
        
        # Diagonal line (perfect prediction)
        val_min = min(df_method[val_col].min(), df_method[pred_col].min())
        val_max = max(df_method[val_col].max(), df_method[pred_col].max())
        ax.plot([val_min, val_max], [val_min, val_max], 
               color=diagonal_color, linestyle=diagonal_style, lw=diagonal_width)
        
        # Diagonal line (perfect prediction)
        val_min = min(df_method[val_col].min(), df_method[pred_col].min())
        val_max = max(df_method[val_col].max(), df_method[pred_col].max())
        ax.plot([val_min, val_max], [val_min, val_max], 
               color=diagonal_color, linestyle=diagonal_style, lw=diagonal_width)

        # Threshold lines
        ax.axhline(y=thresh, color=threshold_color, linestyle=threshold_style, 
                  lw=threshold_width, alpha=threshold_alpha)
        ax.axvline(x=thresh, color=threshold_color, linestyle=threshold_style, 
                  lw=threshold_width, alpha=threshold_alpha)

        # Metrics text box
        metrics_text = (f"MAE: {df_metrics['mae'].mean():.2f}\n" +
                       f"MSE: {df_metrics['mse'].mean():.2f}\n" +
                       f"RMSE: {np.sqrt(df_metrics['mse'].mean()):.2f}\n" +
                       f"$R^2$: {df_metrics['r2'].mean():.2f}\n" +
                       f"$\\rho$: {df_metrics['rho'].mean():.2f}\n" +
                       f"Precision: {precision:.2f}\n" +
                       f"Recall: {recall:.2f}\n" +
                       f"AUC: {roc_auc:.2f}")
        
        if i == 0:
            # Left plot: place box in top-right corner
            pos = (0.05, 0.95)
        else:
            # Right plot: place box in mid-left corner
            pos = (0.05, 0.72)
        
        ax.text(*pos, metrics_text, transform=ax.transAxes,
               verticalalignment='top', fontsize=font_size-3,
               # Add a contoured box around the text
               bbox=dict(boxstyle='round', facecolor='white', alpha=metrics_box_alpha, edgecolor='black', linewidth=0.5))

        # Axis labels
        ax.set_xlabel(f'Predicted {ax_name}', fontsize=xylabel_font_size, fontweight=xlabel_fontweight)
        ax.set_ylabel(f'Measured {ax_name}', fontsize=xylabel_font_size, fontweight=xlabel_fontweight)
        ax.tick_params(axis='both', labelsize=tick_font_size)
        ax.grid(axis='both', alpha=grid_alpha)
    
    # Add subplot label in the top-left corner
    if subplot_label:
        axes[0].text(-0.2, 0.95, subplot_label,
                     transform=axes[0].transAxes,
                     fontsize=font_size * 1.5, fontweight='bold')

    return fig, axes


dmax_methods = {
    # 'XGB Cell-Desc E3-Ord Mol-Desc POI-Vec Time': 'XGB Cell-Desc E3-Ord POI-Vec Mol-Desc',
    # 'XGB-DMAX Cell-Text E3-Ord Mol-Desc POI-Vec Time': 'XGB-DMAX Cell-Text E3-Ord Mol-Desc POI-Vec',  
    'XGB-DMAX Cell-Text E3-OneHot Mol-Desc POI-Vec Time': 'XGB-DMAX Cell-Text E3-OneHot Mol-Desc POI-Vec',  
}
dc50_methods = {
    # 'XGB Cell-Desc E3-Ord FP Mol-Desc POI-Vec Time': 'XGB Cell-Desc E3-Ord POI-Vec Mol-Desc FP',
    # 'XGB-DC50 Cell-Text E3-Ord FP Mol-Desc POI-Vec Time': 'XGB-DC50 Cell-Text E3-Ord FP Mol-Desc POI-Vec',
    'MLP-DC50 Cell-OneHot E3-OneHot Mol-Desc POI-OneHot Time': 'MLP-DC50 Cell-OneHot E3-OneHot Mol-Desc POI-OneHot',
}

subset = results_df[results_df['set'] == 'test'].copy()

# Replace method names according to the mapping
df_dmax = subset[
    (subset['method'].isin(dmax_methods.keys())) &
    (subset['task'] == 'Dmax')
].copy()
df_dmax['method'] = df_dmax['method'].map(dmax_methods)

df_dc50 = subset[
    (subset['method'].isin(dc50_methods.keys())) &
    (subset['task'] == 'DC50')
].copy()
df_dc50['method'] = df_dc50['method'].map(dc50_methods)

fig, axes = make_scatterplot_pair(
    df_dc50,
    df_dmax,
    val_col="target",
    pred_col="pred",
    thresh_left=classification_cutoffs['DC50'],
    thresh_right=classification_cutoffs['Dmax'],
    cycle_col="fold",
    group_col="method",
    axis_left="$pDC_{50}$",
    axis_right="$D_{\\mathrm{max}}$",
    # Use full text width for side-by-side plots
    tex_width=506.295,  # ACM text width
    # tex_width=241.14749,  # ACM column width
    fraction=1.0,       # Use full width
    font_size=10,        # ACM body font size
    scatter_size=7,     # Slightly larger points for visibility
    scatter_alpha=0.5,    # More opaque points for better visibility
)

# Save WITHOUT bbox_inches='tight' to preserve exact dimensions
plt.savefig(plot_dir / "scatterplot_pair_dmax_pdc50.pdf")
plt.savefig(plot_dir / "scatterplot_pair_dmax_pdc50.svg")
plt.show()